In [24]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox
import polars as pl

In [25]:
ROOT_PATH = Path(".").resolve().absolute()
DATASET_PATH = ROOT_PATH / "sumo/simulations/ohare-chicago-junctionless/output/fcd.parquet"
NETWORK_PATH = ROOT_PATH / "networks/graphml/ohare_network.graphml"

In [26]:
G = ox.load_graphml(NETWORK_PATH)
edges_gdf = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True).to_crs(epsg=4326)
edges_gdf


,,,osmid,highway,lanes,name,oneway,ref,reversed,length,geometry,maxspeed,bridge
u,v,key,,,,,,,,,,,
1153867776,29839280,0,985432262,primary,3,Mannheim Road,True,US 12;US 45,False,22.870702,"LINESTRING (-87.87914 41.9887, -87.87925 41.98...",NaN,NaN
29839280,1153867817,0,985432262,primary,3,Mannheim Road,True,US 12;US 45,False,38.594048,"LINESTRING (-87.87925 41.98889, -87.87943 41.9...",NaN,NaN
1153867781,1153867758,0,11537207,motorway_link,1,NaN,True,NaN,False,30.909291,"LINESTRING (-87.88111 41.98211, -87.88126 41.9...",NaN,NaN
1153867758,102856723,0,11537207,motorway_link,1,NaN,True,NaN,False,19.860727,"LINESTRING (-87.88126 41.98185, -87.88133 41.9...",NaN,NaN
29786118,4686227143,0,320340442,tertiary,3,Bessie Coleman Drive,True,NaN,False,95.216379,"LINESTRING (-87.88573 41.97871, -87.88573 41.9...",30 mph,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29785981,5037315526,0,31296557,unclassified,2,O'Hare International Terminal Departures,True,NaN,False,8.552609,"LINESTRING (-87.89083 41.97663, -87.89092 41.9...",20 mph,NaN
1153867719,102857949,0,11537330,motorway_link,NaN,NaN,True,NaN,False,15.968346,"LINESTRING (-87.87667 41.97976, -87.87658 41.9...",NaN,NaN
1153867766,102855117,0,19012965,motorway_link,1,NaN,True,NaN,False,22.372636,"LINESTRING (-87.88105 41.98147, -87.88105 41.9...",NaN,NaN


In [27]:
lf = pl.scan_parquet(DATASET_PATH)
df = lf.collect()
df.to_pandas()

,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
0,0,"[-87.8856713296918, 41.99494072305388]",0.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885671,41.994941
1,0,"[-87.88567733180858, 41.994936271618215]",1.0,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885677,41.994936
2,0,"[-87.88568911243644, 41.994927534580114]",1.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885689,41.994928
3,0,"[-87.88570568973026, 41.99491498626672]",2.0,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885706,41.994915
4,0,"[-87.88572492532072, 41.994900100759565]",2.5,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885725,41.994900
...,...,...,...,...,...,...,...,...,...,...
859435,1702,"[-87.88874921463292, 41.99804074104869]",4081.5,:10021303714_0_0,:10021303714_0_0,1094214001,False,node_10021303714,-87.888749,41.998041
859436,1702,"[-87.88875558331904, 41.99798545992872]",4082.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997985
859437,1702,"[-87.88875619867935, 41.997933337316766]",4082.5,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997933
859438,1702,"[-87.88875677920512, 41.99788060455132]",4083.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888757,41.997881


In [28]:
df.describe()

statistic,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
str,f64,f64,f64,str,str,str,f64,str,f64,f64
"""count""",859440.0,859440.0,859440.0,"""859440""","""859440""","""859035""",859440.0,"""859440""",859440.0,859440.0
"""null_count""",0.0,0.0,0.0,"""0""","""0""","""405""",0.0,"""0""",0.0,0.0
"""mean""",879.541221,null,1972.413852,null,null,null,0.058079,null,-87.889104,41.984193
"""std""",507.302028,null,1029.852968,null,null,null,null,null,0.007366,0.006624
"""min""",0.0,null,0.5,null,null,null,0.0,null,-87.906272,41.973193
"""25%""",443.0,null,1092.5,null,null,null,null,null,-87.892889,41.978784
"""50%""",872.0,null,1965.5,null,null,null,null,null,-87.885743,41.982073
"""75%""",1323.0,null,2851.5,null,null,null,null,null,-87.885582,41.98961
"""max""",1758.0,null,4083.5,null,null,null,1.0,null,-87.876231,41.999815


In [29]:
result = edges_gdf.reset_index().set_index("osmid").loc[321574770.0][["u", "v"]].values.tolist()
result

[np.int64(7710696745), np.int64(5493833658)]

In [30]:
from functools import lru_cache


@lru_cache(maxsize=None)
def ensure_connection(
    G_road: nx.MultiDiGraph, source: int, target: int
) -> list[int] | None:
    try:
        path = nx.shortest_path(G_road, source=source, target=target, weight="length")
        edges_path = list((x, y, 0) for x, y in zip(path[:-1], path[1:]))
        edge_data = list(G.edges[edge]["osmid"] for edge in edges_path)
        res = []
        for ed in edge_data:
            if ed not in res:
                res.append(ed)
        return res

    except nx.NetworkXNoPath:
        return None

In [31]:
df.filter(pl.col("node_mapped_id").cat.starts_with("node_"))

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
0,"[-87.885748, 41.994882]",3.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885748,41.994882
0,"[-87.885771, 41.994857]",3.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885771,41.994857
0,"[-87.885789, 41.994827]",4.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885789,41.994827
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888708, 41.999024]",4070.0,""":1350812521_6_0""",""":1350812521_6_0""","""1094498007""",false,"""node_1350812521""",-87.888708,41.999024
1702,"[-87.888756, 41.998723]",4075.0,""":10024343559_0_0""",""":10024343559_0_0""","""1094214000""",false,"""node_10024343559""",-87.888756,41.998723
1702,"[-87.888723, 41.998676]",4075.5,""":10024343559_0_0""",""":10024343559_0_0""","""1094214000""",false,"""node_10024343559""",-87.888723,41.998676


In [32]:
from typing import cast

import pandas as pd


indexed_osm_edges = edges_gdf.reset_index().set_index("osmid")

node_df = df.filter(pl.col("node_mapped_id").cat.starts_with("node_"))

# pairs: set[tuple[int, int]] = set()

# for row in node_df.iter_rows(named=True):
#     next_id = row.get("edge_id")
#     if next_id is None:
#         continue
#     next_id = int(next_id)

#     node_mapped_id = row.get("node_mapped_id")
#     if node_mapped_id is None:
#         continue

#     node_mapped_id = row.get("node_mapped_id")
#     if node_mapped_id is None:
#         print(f"Skipping row with missing node_mapped_id: {row}")
#         continue

#     # node_mapped_id can be like "node_1234" or already an int/np.int64
#     if isinstance(node_mapped_id, str) and node_mapped_id.startswith("node_"):
#         node_mapped_id = int(node_mapped_id.replace("node_", ""))
#     else:
#         node_mapped_id = int(node_mapped_id)


dfs_to_add = []
for row in node_df.iter_rows(named=True):
    # skip rows without an edge_id
    next_id = row.get("edge_id")
    if next_id is None:
        # edge_id is unmapped for this row; skip it
        # optionally you can log or collect these rows for inspection
        print(f"Skipping row with missing edge_id: {row["raw_lane_id"]}")
        continue

    # ensure next_id is an int (handles numpy int types as well)
    next_id = int(next_id)

    node_mapped_id = row.get("node_mapped_id")
    if node_mapped_id is None:
        print(f"Skipping row with missing node_mapped_id: {row}")
        continue

    # node_mapped_id can be like "node_1234" or already an int/np.int64
    if isinstance(node_mapped_id, str) and node_mapped_id.startswith("node_"):
        node_mapped_id = int(node_mapped_id.replace("node_", ""))
    else:
        node_mapped_id = int(node_mapped_id)

    try:
        sel = indexed_osm_edges.loc[next_id, ["u", "v"]]  # type: ignore
        if isinstance(sel, (pd.Series,)):
            sel_df = sel.to_frame().T
        else:
            sel_df = sel
        edge = sel_df.values.tolist()
        if len(edge) == 0:
            print(f"No edge found for node {next_id}")
            continue

        if node_mapped_id and all((node_mapped_id not in e) for e in edge):
            # if u != node_mapped_id and node_mapped_id :
            neighbors = G.neighbors(node_mapped_id)
            
            u, v = next(((u, v) for u, v in edge if u in neighbors), edge[0])

            connect_edges = ensure_connection(G, node_mapped_id, u)
            if connect_edges:
                # insert new edges into df
                new_row = row.copy()
                del new_row["edge_id"]
                del new_row["time"]

                new_df = pl.DataFrame(
                    [
                        {
                            **new_row,
                            "edge_id": str(connect_edge),
                            "time": float(row["time"] + (i * 0.01)),
                        }
                        for i, connect_edge in enumerate(connect_edges)
                    ]
                )
                new_df = new_df.with_columns(
                    pl.col("time").cast(pl.Float64),
                    pl.col("raw_lane_id").cast(pl.Categorical),
                    pl.col("vehicle_id").cast(pl.Int64),
                    pl.col("geo_position").cast(pl.Array(pl.Float64, shape=2)),
                    pl.col("edge_id").cast(pl.Categorical),
                    pl.col("node_mapped_id").cast(pl.Categorical),
                    pl.col("mapped_lane_id").cast(pl.Categorical),
                )
                dfs_to_add.append(new_df)
                df = df.remove(pl.col("raw_lane_id") == row["raw_lane_id"])  # type: ignore

    except KeyError:
        print(f"KeyError: {node_mapped_id}")

Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :6378331511_0_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :11038570643_0_0
Skipping row with missing edge_id: :11038570643_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793878899_2_0
Skipping row with missing edge_

In [41]:
df_to_add = pl.DataFrame(
        {
            "time": pl.Int64,
            "vehicle_id": pl.Int64,
            "geo_position": pl.Array(pl.Float64, shape=2),
            "edge_id": pl.Categorical,
            "mapped_lane_id": pl.Categorical,
            "node_mapped_id": pl.Categorical,
            "raw_lane_id": pl.Categorical,
        }
    )

if dfs_to_add:
    df_to_add = pl.concat(dfs_to_add)
    df_to_add = df_to_add.with_columns(
        pl.col("time").cast(pl.Float64),
        pl.col("vehicle_id").cast(pl.Int64),
        pl.col("geo_position").cast(pl.Array(pl.Float64, shape=2)),
        
        pl.col("edge_id").cast(pl.Categorical),
        pl.col("mapped_lane_id").cast(pl.Categorical),
        pl.col("node_mapped_id").cast(pl.Categorical),
        pl.col("raw_lane_id").cast(pl.Categorical),
    )

In [42]:
df_to_add.describe()

statistic,vehicle_id,geo_position,raw_lane_id,mapped_lane_id,reversed,node_mapped_id,lon,lat,edge_id,time
str,f64,f64,str,str,f64,str,f64,f64,str,f64
"""count""",25596.0,25596.0,"""25596""","""25596""",25596.0,"""25596""",25596.0,25596.0,"""25596""",25596.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,0.0,"""0""",0.0
"""mean""",907.876973,null,null,null,0.0,null,-87.889673,41.983587,null,2021.685422
"""std""",509.922666,null,null,null,null,null,0.006882,0.006597,null,1037.783419
"""min""",2.0,null,null,null,0.0,null,-87.905965,41.976659,null,40.5
"""25%""",493.0,null,null,null,null,null,-87.899109,41.979059,null,1167.0
"""50%""",927.0,null,null,null,null,null,-87.885784,41.980161,null,2023.0
"""75%""",1368.0,null,null,null,null,null,-87.885605,41.990362,null,2947.02
"""max""",1758.0,null,null,null,0.0,null,-87.879078,41.995177,null,3999.0


In [43]:
df_to_add = df_to_add.select(df.columns)

concated_df = pl.concat([df, df_to_add], how="vertical")
concated_df = concated_df.sort(["time", "vehicle_id"])
concated_df

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885671, 41.994941]",0.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885671,41.994941
0,"[-87.885677, 41.994936]",1.0,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885677,41.994936
0,"[-87.885689, 41.994928]",1.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885689,41.994928
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888749, 41.998041]",4081.5,""":10021303714_0_0""",""":10021303714_0_0""","""1094214001""",false,"""node_10021303714""",-87.888749,41.998041
1702,"[-87.888756, 41.997985]",4082.0,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997985
1702,"[-87.888756, 41.997933]",4082.5,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997933


In [44]:
c = concated_df.filter(pl.col("vehicle_id") == 1001).to_pandas().sort_values(["vehicle_id", "time"])
c

,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
0,1001,"[-87.88580060152805, 41.97764234868195]",2092.0,1238156320_0,1238156320_0,1238156320,False,1238156320,-87.885801,41.977642
1,1001,"[-87.88576324022672, 41.97763372198496]",2092.5,1238156320_1,1238156320_1,1238156320,False,1238156320,-87.885763,41.977634
2,1001,"[-87.88572655353623, 41.977620687177506]",2093.0,1238156320_2,1238156320_2,1238156320,False,1238156320,-87.885727,41.977621
3,1001,"[-87.88572877714086, 41.977608018398676]",2093.5,1238156320_2,1238156320_2,1238156320,False,1238156320,-87.885729,41.977608
4,1001,"[-87.88573159751046, 41.97759194960051]",2094.0,1238156320_2,1238156320_2,1238156320,False,1238156320,-87.885732,41.977592
...,...,...,...,...,...,...,...,...,...,...
594,1001,"[-87.8887181048239, 41.998918873263484]",2387.5,1094498007_1,1094498007_1,1094498007,False,1094498007,-87.888718,41.998919
595,1001,"[-87.88872077008891, 41.99888294985146]",2388.0,1094498007_1,1094498007_1,1094498007,False,1094498007,-87.888721,41.998883
596,1001,"[-87.88872216880448, 41.99884171195431]",2388.5,1094498007_1,1094498007_1,1094498007,False,1094498007,-87.888722,41.998842
597,1001,"[-87.88872193244057, 41.9987950823738]",2389.0,1094498007_1,1094498007_1,1094498007,False,1094498007,-87.888722,41.998795


In [45]:
concated_df.write_parquet(DATASET_PATH.with_name("fcd_resolved.parquet"), compression="zstd")

In [46]:

lf2 = lf.with_columns(
    pl.col("mapped_lane_id").cast(pl.String)
    .str.extract(r"(-?\d+)(?:.*)", 1)
    .alias("edge_id")
)

lf2 = lf2.with_columns(
    pl.col("edge_id").str.starts_with("-").alias("reversed"),
    pl.col("edge_id").str.replace("-", ""),
)

is_node = pl.col("edge_id").cast(pl.Int64).is_in(G.nodes())

lf2 = lf2.with_columns(
    pl.when(is_node)
    .then(pl.lit("node_") + pl.col("edge_id"))
    .otherwise(pl.col("edge_id"))
    .alias("edge_id")
)

lf2 = lf2.with_columns(
    pl.col("raw_lane_id").cast(pl.Categorical),
    pl.col("mapped_lane_id").cast(pl.Categorical),
    pl.col("edge_id").cast(pl.Categorical)
)

lf2 = lf2.with_columns(
    pl.col("edge_id").alias("node_mapped_id")
)

lf2 = lf2.with_columns(
    pl.when(~pl.col("edge_id").cat.starts_with("node_"))
    .then(pl.col("edge_id"))
    .otherwise(None)
    .alias("edge_id_valid")
)

lf2 = lf2.with_columns(
    pl.col("edge_id_valid")
    .backward_fill()
    .over("vehicle_id")
    .alias("edge_id")
)

lf2.sort(["vehicle_id", "time"])

df2 = lf2.collect()
pd_df2 = df2.to_pandas().sort_values(["vehicle_id", "time"])
pd_df2

,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat,edge_id_valid
0,0,"[-87.8856713296918, 41.99494072305388]",0.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885671,41.994941,1188647984
1,0,"[-87.88567733180858, 41.994936271618215]",1.0,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885677,41.994936,1188647984
2,0,"[-87.88568911243644, 41.994927534580114]",1.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885689,41.994928,1188647984
3,0,"[-87.88570568973026, 41.99491498626672]",2.0,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885706,41.994915,NaN
4,0,"[-87.88572492532072, 41.994900100759565]",2.5,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885725,41.994900,NaN
...,...,...,...,...,...,...,...,...,...,...,...
849776,1758,"[-87.88204500820018, 41.98479676583491]",3760.5,-4685903_0,-4685903_0,4685903,True,4685903,-87.882045,41.984797,4685903
849830,1758,"[-87.8819512028677, 41.98479704745202]",3761.0,-4685903_0,-4685903_0,4685903,True,4685903,-87.881951,41.984797,4685903
849884,1758,"[-87.88185635890108, 41.98479733210934]",3761.5,-4685903_0,-4685903_0,4685903,True,4685903,-87.881856,41.984797,4685903
849938,1758,"[-87.88176341113001, 41.98478515335567]",3762.0,-4685903_0,-4685903_0,4685903,True,4685903,-87.881763,41.984785,4685903


In [47]:
[(u, v) for u, v, data in G.edges(data=True) if data.get("osmid") == 474730115]

[(7710696746, 4686227120),
 (8872546867, 10970085837),
 (10970085837, 10970085838),
 (10970085838, 7710696746)]